In [1]:
import lightgbm as lgb
import numpy as np
import optuna
import polars as pl
from sklearn.model_selection import train_test_split

In [2]:
RANDOM_SEED = 1000
df = pl.read_parquet("data/1L83:p2rank:1.fingerprints.parquet")

In [3]:
FEATURE_NAMES = [
    "heavy_atom_count",
    "molecular_weight",
    "calculated_partition_coefficient",
    "calculated_distribution_coefficient",
    "topological_polar_surface_area",
    "hydrogen_bond_donors",
    "pka",
]

LABEL_NAME = "affinity_kcal_mol"

x_scalars = df.select(FEATURE_NAMES).to_numpy()

x_morgan = np.array(df["morgan_fingerprint"].to_list())
x_e3fp = np.array(df["e3fp"].to_list())
x_usrcat = np.array(df["usrcat"].to_list())
x_whim = np.array(df["whim"].to_list())
x_getaway = np.array(df["getaway"].to_list())
x_pharmacophore = np.array(df["pharmacophore_3d"].to_list())

x_all_fingerprints = np.hstack(
    [
        x_morgan,
        x_e3fp,
        x_usrcat,
        x_whim,
        x_getaway,
        x_pharmacophore,
    ]
)

x = np.hstack([x_scalars, x_all_fingerprints])

y = df[LABEL_NAME].to_numpy()

In [4]:
TRAIN_FRACTION = 0.75

x_train, x_test, y_train, y_test = train_test_split(
    x, y, train_size=TRAIN_FRACTION, random_state=RANDOM_SEED
)

In [5]:
from surrogate_model.optuna import make_objective

NUM_TRIALS = 50

PRIMARY_METRIC = "bedroc"

sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)

study.optimize(
    make_objective(
        X=x_train,
        y=y_train,
        n_splits=5,
        primary_metric=PRIMARY_METRIC,
        random_seed=RANDOM_SEED,
    ),
    n_trials=NUM_TRIALS,
    show_progress_bar=True,
)

[I 2026-09-10 00:30:28,668] A new study created in memory with name: no-name-d066c109-c868-45a4-b3a6-c454a05f47a3


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-09-10 00:32:30,375] Trial 0 finished with value: 0.21542606613599072 and parameters: {'num_leaves': 173, 'max_depth': 4, 'learning_rate': 0.22592582730543753, 'n_estimators': 1016, 'min_child_samples': 88, 'subsample': 0.60616634046136, 'colsample_bytree': 0.5203548123845445, 'reg_alpha': 3.7562124870680294e-05, 'reg_lambda': 1.2536888868437446e-06}. Best is trial 0 with value: 0.21542606613599072.
[I 2026-09-10 00:36:35,096] Trial 1 finished with value: 0.24165091032001823 and parameters: {'num_leaves': 218, 'max_depth': 5, 'learning_rate': 0.06905371732106186, 'n_estimators': 845, 'min_child_samples': 22, 'subsample': 0.87176970729607, 'colsample_bytree': 0.5347910404849773, 'reg_alpha': 0.9290409121444837, 'reg_lambda': 3.7480000930162687}. Best is trial 1 with value: 0.24165091032001823.
[I 2026-09-10 00:50:11,611] Trial 2 finished with value: 0.2810497370552242 and parameters: {'num_leaves': 240, 'max_depth': 7, 'learning_rate': 0.001179752983913515, 'n_estimators': 1966, 

In [6]:
best_params = study.best_params
best_params.update({"random_state": RANDOM_SEED})

final_model = lgb.LGBMRegressor(**best_params, deterministic=True, force_row_wise=True)
final_model.fit(
    x_train,
    y_train,
    eval_X=x_test,
    eval_y=y_test,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)],
)

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1778]	valid_0's l2: 156.962


,num_leaves,173
,max_depth,11
,learning_rate,0.001022285456997112
,n_estimators,1778
,min_child_samples,61
,subsample,0.6718807799877679
,colsample_bytree,0.7973606247486545
,reg_alpha,2.4517081529270666
,reg_lambda,0.0001024362410488648
,random_state,1000
,deterministic,True


In [ ]:
from skfp.metrics import bedroc_score, enrichment_factor, spearman_correlation

y_pred = np.asarray(final_model.predict(x_test))

active_quantile = 0.05
affinity_threshold = np.quantile(y_test, active_quantile)
y_true_binary = (y_test <= affinity_threshold).astype(int)
scores = -y_pred

results = {
    "spearman": float(spearman_correlation(y_test, y_pred)),
    "bedroc": float(bedroc_score(y_true_binary, scores, alpha=20.0)),
    "enrichment_factor_1_percent": float(
        enrichment_factor(y_true_binary, scores, 0.01)
    ),
    "enrichment_factor_5_percent": float(
        enrichment_factor(y_true_binary, scores, 0.05)
    ),
    "enrichment_factor_10_percent": enrichment_factor(y_true_binary, scores, 0.1),
}

results

{'spearman': 0.4233694860847111,
 'enrichment_factor_1_percent': 7.5533152129149155,
 'enrichment_factor_5_percent': 5.157847478927641,
 'enrichment_factor_10_percent': 3.6682966983752623,
 'bedroc': 0.32854577307105565}

| train size | train metric | trials | spearman | bedroc | enrichment 1 percent | enrichment 5 percent | enrichment 10 percent |
| - | - | - | - | - | - | - | - |
| 10000 | bedroc | 20 | 0.406 | 0.331 | 8.83 | 4.93 | 3.52 |
| 50000 | bedroc | 20 | - | 0.372 | 11.05 | 5.65 | 3.82 |
| 75000 | bedroc | 50 | 0.423 | 0.329 | 7.55 | 5.16 | 3.67 |

Targets:

Spearman ρ > 0.5

EF@1% > 10-20 is considered reasonably strong for early enrichment in virtual screening
EF@1% > 30-50 is very good, approaching what you'd see from decent docking scores themselves
